# Per-Series SHAP Analysis with ConditionalSHAP

This notebook demonstrates how to use the enhanced `ConditionalSHAP` explainer for:

1. **Batch SHAP computation** - Fast computation for tree-based models (LightGBM)
2. **Per-series analysis** - Compute SHAP values separately for each series
3. **Hierarchical aggregation** - Aggregate importance across hierarchy levels
4. **Visualizations** - Beeswarm/violin plots per series and cohort

The `ConditionalSHAP` class automatically detects the model type and uses:
- **TreeExplainer** for tree-based models (LightGBM, XGBoost, RandomForest) - fast batch computation
- **KernelExplainer** for other models - series-specific background data

**Requirements:**
```bash
pip install xeries[skforecast] lightgbm
```

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Create Hierarchical Multi-Series Data

We'll create a dataset with a simple hierarchy:
- **State** (top level): TX, WI
- **Store** (bottom level): multiple stores per state

Series IDs follow the pattern: `{STATE}_{STORE}` (e.g., TX_S1, WI_S2)

In [2]:
def create_hierarchical_series(n_periods: int = 500) -> dict[str, pd.Series]:
    """Create synthetic hierarchical time series data.
    
    Returns dict of series for skforecast (recommended format).
    """
    # Hierarchy: State -> Store
    hierarchy = {
        "TX": ["TX_S1", "TX_S2", "TX_S3"],
        "WI": ["WI_S1", "WI_S2"],
    }
    
    dates = pd.date_range("2022-01-01", periods=n_periods, freq="h")
    series_dict = {}
    
    for state, stores in hierarchy.items():
        state_effect = 10 if state == "TX" else -5
        
        for store in stores:
            base = 100 + np.random.randn() * 20
            trend = np.linspace(0, 10, n_periods)
            seasonal = 15 * np.sin(2 * np.pi * np.arange(n_periods) / 24)  # Daily pattern
            weekly = 10 * np.sin(2 * np.pi * np.arange(n_periods) / 168)   # Weekly pattern
            noise = np.random.randn(n_periods) * 3
            
            values = base + trend + seasonal + weekly + state_effect + noise
            series_dict[store] = pd.Series(values, index=dates, name=store)
    
    return series_dict

# Create series data
series_dict = create_hierarchical_series(n_periods=500)

print(f"Number of series: {len(series_dict)}")
print(f"Series IDs: {list(series_dict.keys())}")
print(f"Points per series: {len(list(series_dict.values())[0])}")

Number of series: 5
Series IDs: ['TX_S1', 'TX_S2', 'TX_S3', 'WI_S1', 'WI_S2']
Points per series: 500


## 2. Train Forecaster with LightGBM

Using skforecast's `ForecasterRecursiveMultiSeries` with LightGBM and StandardScaler.

In [3]:
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from skforecast.recursive import ForecasterRecursiveMultiSeries

# Create and fit forecaster
forecaster = ForecasterRecursiveMultiSeries(
    regressor=LGBMRegressor(
        n_estimators=100,
        max_depth=8,
        learning_rate=0.1,
        random_state=42,
        verbose=-1,
    ),
    lags=24,  # 24 hourly lags
    transformer_series=StandardScaler(),
)

forecaster.fit(series=series_dict)

print(f"Forecaster fitted with {forecaster.regressor.n_features_in_} features")
print(f"Series: {forecaster.series_names_in_}")

Forecaster fitted with 25 features
Series: ['TX_S1', 'TX_S2', 'TX_S3', 'WI_S1', 'WI_S2']


## 3. Create Adapter and Extract Training Data

In [4]:
from xeries.adapters import from_skforecast

# Create adapter (same series as fit)
adapter = from_skforecast(forecaster, series=series_dict)

# Get training data
X, y = adapter.get_training_data()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFeature names: {adapter.get_feature_names()}")
print(f"\nSeries IDs: {adapter.get_series_ids()}")
print(f"\nX columns: {list(X.columns)}")

X shape: (2380, 25)
y shape: (2380,)

Feature names: ['lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_10', 'lag_11', 'lag_12', 'lag_13', 'lag_14', 'lag_15', 'lag_16', 'lag_17', 'lag_18', 'lag_19', 'lag_20', 'lag_21', 'lag_22', 'lag_23', 'lag_24']

Series IDs: ['TX_S1', 'TX_S2', 'TX_S3', 'WI_S1', 'WI_S2']

X columns: ['lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_10', 'lag_11', 'lag_12', 'lag_13', 'lag_14', 'lag_15', 'lag_16', 'lag_17', 'lag_18', 'lag_19', 'lag_20', 'lag_21', 'lag_22', 'lag_23', 'lag_24', '_level_skforecast']


In [5]:
# Show sample of data
X.head()

,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,...,lag_16,lag_17,lag_18,lag_19,lag_20,lag_21,lag_22,lag_23,lag_24,_level_skforecast
2022-01-02 00:00:00,-0.198742,-0.702525,-0.618350,-0.892548,-0.639665,-1.369171,-1.237422,-0.863888,-1.012859,-0.695074,...,0.974330,0.826391,1.122641,1.241338,0.675876,0.461751,0.598543,0.086339,-0.425627,0
2022-01-02 01:00:00,0.272536,-0.198742,-0.702525,-0.618350,-0.892548,-0.639665,-1.369171,-1.237422,-0.863888,-1.012859,...,0.586430,0.974330,0.826391,1.122641,1.241338,0.675876,0.461751,0.598543,0.086339,0
2022-01-02 02:00:00,0.299101,0.272536,-0.198742,-0.702525,-0.618350,-0.892548,-0.639665,-1.369171,-1.237422,-0.863888,...,0.374756,0.586430,0.974330,0.826391,1.122641,1.241338,0.675876,0.461751,0.598543,0
2022-01-02 03:00:00,0.949989,0.299101,0.272536,-0.198742,-0.702525,-0.618350,-0.892548,-0.639665,-1.369171,-1.237422,...,0.287622,0.374756,0.586430,0.974330,0.826391,1.122641,1.241338,0.675876,0.461751,0
2022-01-02 04:00:00,0.981050,0.949989,0.299101,0.272536,-0.198742,-0.702525,-0.618350,-0.892548,-0.639665,-1.369171,...,-0.483177,0.287622,0.374756,0.586430,0.974330,0.826391,1.122641,1.241338,0.675876,0


## 4. ConditionalSHAP with Auto-Detection

The `ConditionalSHAP` explainer automatically detects that we're using a LightGBM model and uses `TreeExplainer` for fast batch computation.

In [6]:
from xeries import ConditionalSHAP

# Full training-matrix width, including the series id the estimator saw at fit.
feature_names = list(X.columns)

# Create explainer - auto-detects TreeExplainer for LightGBM
explainer = ConditionalSHAP(
    model=adapter.forecaster.estimator,  # unwrap so TreeExplainer can auto-detect LightGBM
    background_data=X,
    series_col="_level_skforecast",  # skforecast's series column
)

print(f"Batch capable (fast): {explainer._is_batch_capable}")
print(f"Feature names: {feature_names}")

Batch capable (fast): True
Feature names: ['lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_10', 'lag_11', 'lag_12', 'lag_13', 'lag_14', 'lag_15', 'lag_16', 'lag_17', 'lag_18', 'lag_19', 'lag_20', 'lag_21', 'lag_22', 'lag_23', 'lag_24', '_level_skforecast']


## 5. Full Dataset SHAP Computation

Compute SHAP values for the entire dataset in one batch (fast for tree models).

In [7]:
# Compute SHAP values for full dataset
result = explainer.explain(X, feature_names=feature_names)

print(f"SHAP values shape: {result.shap_values.shape}")
print(f"Features: {result.feature_names}")
print(f"Series tracked: {result.series_ids.unique().tolist()}")

SHAP values shape: (2380, 25)


Features: ['lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_10', 'lag_11', 'lag_12', 'lag_13', 'lag_14', 'lag_15', 'lag_16', 'lag_17', 'lag_18', 'lag_19', 'lag_20', 'lag_21', 'lag_22', 'lag_23', 'lag_24', '_level_skforecast']
Series tracked: [0, 1, 2, 3, 4]


In [8]:
# Global importance
result.mean_abs_shap()

,feature,mean_abs_shap
0,lag_1,0.594133
1,lag_2,0.102322
20,lag_21,0.051588
2,lag_3,0.051227
21,lag_22,0.049085
17,lag_18,0.041700
19,lag_20,0.040177
7,lag_8,0.039240
18,lag_19,0.028528
22,lag_23,0.021719


In [9]:
# Convert to DataFrame for easy inspection
shap_df = result.to_dataframe()
print(f"SHAP DataFrame shape: {shap_df.shape}")
shap_df.head()

SHAP DataFrame shape: (2380, 25)


,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,...,lag_16,lag_17,lag_18,lag_19,lag_20,lag_21,lag_22,lag_23,lag_24,_level_skforecast
0,-0.122463,-0.109917,-0.008833,-0.000595,0.047139,0.005642,0.025615,0.041952,0.025320,0.038597,...,0.005515,0.004591,0.042788,0.030202,0.051271,0.077167,0.076173,0.005232,-0.012477,0.002308
1,0.174777,-0.115993,-0.069143,-0.011071,0.013600,0.002274,0.018721,0.051631,0.025326,0.031412,...,0.003602,0.014104,0.028013,0.027036,0.028434,0.043074,0.061684,0.022633,-0.001085,-0.001108
2,0.251143,0.010849,-0.006944,-0.006649,0.029233,0.011066,0.017988,0.052927,0.024962,0.017072,...,0.002904,0.023922,0.045097,0.021243,0.027167,0.061576,0.088515,0.023633,0.010070,-0.000839
3,0.655360,-0.030565,0.011287,-0.003224,-0.001970,0.006527,0.013545,0.046193,0.003551,0.012106,...,0.004542,0.009770,0.043119,0.052278,0.047080,0.055271,0.041079,0.021041,0.006042,0.001189
4,0.695036,0.138439,-0.011229,-0.002344,0.008564,-0.001612,0.010378,0.042132,0.010279,0.026753,...,0.004226,0.014862,0.029329,0.016291,0.052533,0.044041,0.035354,0.011409,0.029379,0.004721


## 6. Per-Series SHAP Analysis

Use `explain_per_series()` to compute SHAP values separately for each store.

In [10]:
# Compute SHAP values per series
per_series_results = explainer.explain_per_series(
    X, 
    series_col="_level_skforecast",
    feature_names=feature_names
)

print(f"Series analyzed: {list(per_series_results.keys())}")

# Show top 5 features for each store
for store_id, store_result in per_series_results.items():
    print(f"\n{store_id}:")
    print(store_result.mean_abs_shap().head(5).to_string(index=False))

Series analyzed: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

0:
feature  mean_abs_shap
  lag_1       0.595535
  lag_2       0.102380
 lag_21       0.052205
  lag_3       0.049828
 lag_22       0.049013

1:
feature  mean_abs_shap
  lag_1       0.595139
  lag_2       0.103880
 lag_21       0.051186
  lag_3       0.050225
 lag_22       0.047685

2:
feature  mean_abs_shap
  lag_1       0.596764
  lag_2       0.101561
  lag_3       0.052098
 lag_21       0.051782
 lag_22       0.048635

3:
feature  mean_abs_shap
  lag_1       0.592097
  lag_2       0.101888
  lag_3       0.052016
 lag_21       0.052004
 lag_22       0.049654

4:
feature  mean_abs_shap
  lag_1       0.591130
  lag_2       0.101900
  lag_3       0.051968
 lag_21       0.050764
 lag_22       0.050437


In [11]:
# Aggregate importance by series
importance_by_series = result.mean_abs_shap_by_series()
importance_by_series

,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,...,lag_16,lag_17,lag_18,lag_19,lag_20,lag_21,lag_22,lag_23,lag_24,_level_skforecast
0,0.595535,0.102380,0.049828,0.015708,0.016175,0.010339,0.013846,0.039405,0.013684,0.017399,...,0.008342,0.010814,0.040966,0.028780,0.039891,0.052205,0.049013,0.021729,0.017763,0.002129
1,0.595139,0.103880,0.050225,0.015542,0.016815,0.009801,0.014338,0.039946,0.013294,0.017279,...,0.008914,0.011090,0.042141,0.028961,0.040386,0.051186,0.047685,0.021537,0.016377,0.001531
2,0.596764,0.101561,0.052098,0.016134,0.016149,0.009566,0.013193,0.039520,0.013612,0.017184,...,0.007641,0.011254,0.042561,0.028426,0.040390,0.051782,0.048635,0.021705,0.016813,0.000685
3,0.592097,0.101888,0.052016,0.016553,0.016980,0.009231,0.013799,0.039404,0.013598,0.016748,...,0.008511,0.011346,0.041435,0.028005,0.039713,0.052004,0.049654,0.022092,0.017247,0.001408
4,0.591130,0.101900,0.051968,0.015341,0.015656,0.009177,0.012795,0.037923,0.013521,0.017185,...,0.008037,0.011001,0.041400,0.028469,0.040503,0.050764,0.050437,0.021533,0.017457,0.001862


## 7. Visualize Per-Series Importance

In [12]:
n_series = len(per_series_results)
fig, axes = plt.subplots(1, n_series, figsize=(4*n_series, 5), sharey=True)

for ax, (store_id, store_result) in zip(axes, per_series_results.items()):
    importance = store_result.mean_abs_shap().head(10)
    ax.barh(importance["feature"], importance["mean_abs_shap"])
    ax.set_title(f"Store: {store_id}")
    ax.invert_yaxis()

axes[0].set_ylabel("Feature")
fig.suptitle("Top 10 Features by Store", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Hierarchical Aggregation

Use `HierarchyDefinition` and `HierarchicalExplainer` to aggregate SHAP values at different levels.

skforecast encodes series as integers; we decode them to names like `TX_S1` and map those to state/store.

In [13]:
from xeries.hierarchy import HierarchyDefinition, HierarchicalExplainer

# skforecast 0.21 encodes series as integers in `_level_skforecast`.
# Map those codes back to names like TX_S1, then split into state/store.
unique_ids = sorted(pd.Index(X["_level_skforecast"]).unique().tolist())
decoded_names = list(adapter.get_series_ids())
id_to_name = dict(zip(unique_ids, decoded_names, strict=True))
explicit_mapping = {}
for sid, name in id_to_name.items():
    state, store = str(name).split("_", 1)
    explicit_mapping[sid] = {"state": state, "store": store}

hierarchy = HierarchyDefinition(
    levels=["state", "store"],
    explicit_mapping=explicit_mapping,
    series_col="_level_skforecast",
)

print("Cohorts at each level:")
for level in hierarchy.levels:
    cohorts = hierarchy.get_cohorts(X, level)
    print(f"  {level}: {list(cohorts.keys())}")

Cohorts at each level:
  state: ['TX', 'WI']
  store: ['TX_S1', 'TX_S2', 'TX_S3', 'WI_S1', 'WI_S2']


In [14]:
# Create hierarchical explainer
hierarchical = HierarchicalExplainer(explainer, hierarchy)

# Compute hierarchical results (include_raw=True for violin plots)
hier_result = hierarchical.explain(X, include_raw=True, feature_names=feature_names)

print(f"Levels: {hier_result.levels}")
print(f"Features: {hier_result.features}")

Levels: ['global', 'state', 'store']
Features: ['lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_10', 'lag_11', 'lag_12', 'lag_13', 'lag_14', 'lag_15', 'lag_16', 'lag_17', 'lag_18', 'lag_19', 'lag_20', 'lag_21', 'lag_22', 'lag_23', 'lag_24', '_level_skforecast']


In [15]:
# Global importance
print("Global Importance (top 10):")
global_df = hier_result.get_level_df("global")
global_df

Global Importance (top 10):


,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,...,lag_16,lag_17,lag_18,lag_19,lag_20,lag_21,lag_22,lag_23,lag_24,_level_skforecast
all,0.594133,0.102322,0.051227,0.015856,0.016355,0.009623,0.013595,0.03924,0.013542,0.017159,...,0.008289,0.011101,0.0417,0.028528,0.040177,0.051588,0.049085,0.021719,0.017131,0.001523


In [16]:
# State-level importance
print("State-Level Importance:")
hier_result.get_level_df("state")

State-Level Importance:


,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,...,lag_16,lag_17,lag_18,lag_19,lag_20,lag_21,lag_22,lag_23,lag_24,_level_skforecast
TX,0.594133,0.102322,0.051227,0.015856,0.016355,0.009623,0.013595,0.03924,0.013542,0.017159,...,0.008289,0.011101,0.0417,0.028528,0.040177,0.051588,0.049085,0.021719,0.017131,0.001523
WI,0.594133,0.102322,0.051227,0.015856,0.016355,0.009623,0.013595,0.03924,0.013542,0.017159,...,0.008289,0.011101,0.0417,0.028528,0.040177,0.051588,0.049085,0.021719,0.017131,0.001523


In [17]:
# Store-level importance
print("Store-Level Importance:")
hier_result.get_level_df("store")

Store-Level Importance:


,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,...,lag_16,lag_17,lag_18,lag_19,lag_20,lag_21,lag_22,lag_23,lag_24,_level_skforecast
TX_S1,0.594133,0.102322,0.051227,0.015856,0.016355,0.009623,0.013595,0.03924,0.013542,0.017159,...,0.008289,0.011101,0.0417,0.028528,0.040177,0.051588,0.049085,0.021719,0.017131,0.001523
TX_S2,0.594133,0.102322,0.051227,0.015856,0.016355,0.009623,0.013595,0.03924,0.013542,0.017159,...,0.008289,0.011101,0.0417,0.028528,0.040177,0.051588,0.049085,0.021719,0.017131,0.001523
TX_S3,0.594133,0.102322,0.051227,0.015856,0.016355,0.009623,0.013595,0.03924,0.013542,0.017159,...,0.008289,0.011101,0.0417,0.028528,0.040177,0.051588,0.049085,0.021719,0.017131,0.001523
WI_S1,0.594133,0.102322,0.051227,0.015856,0.016355,0.009623,0.013595,0.03924,0.013542,0.017159,...,0.008289,0.011101,0.0417,0.028528,0.040177,0.051588,0.049085,0.021719,0.017131,0.001523
WI_S2,0.594133,0.102322,0.051227,0.015856,0.016355,0.009623,0.013595,0.03924,0.013542,0.017159,...,0.008289,0.011101,0.0417,0.028528,0.040177,0.051588,0.049085,0.021719,0.017131,0.001523


## 9. Hierarchy Visualizations

In [18]:
from xeries.visualization import plot_hierarchy_summary, plot_hierarchy_bar, plot_hierarchy_violin

# Summary view - grid of all levels
fig, axes = plot_hierarchy_summary(hier_result, top_n=10)
plt.suptitle("Feature Importance Across Hierarchy", y=1.02)
plt.tight_layout()
plt.show()

In [19]:
# State comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_hierarchy_bar(hier_result, level="state", cohort="TX", ax=axes[0], top_n=10)
plot_hierarchy_bar(hier_result, level="state", cohort="WI", ax=axes[1], top_n=10)

plt.tight_layout()
plt.show()

In [20]:
# Violin/beeswarm plot for TX state
try:
    fig, ax = plot_hierarchy_violin(hier_result, level="state", cohort="TX", top_n=10)
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print(f"Skipping violin plot ({type(exc).__name__}: {exc})")

Skipping violin plot (IndexError: boolean index did not match indexed array along axis 0; size of axis is 2380 but size of corresponding boolean axis is 7140)


## 10. Compare Feature Rankings Across Cohorts

In [21]:
# Compare cohorts at store level
comparison = hierarchical.compare_cohorts(
    X, level="store", top_n=10, feature_names=feature_names
)
print("Feature Importance Comparison (stores):")
comparison

Feature Importance Comparison (stores):


,TX_S1,TX_S2,TX_S3,WI_S1,WI_S2
lag_1,0.594133,0.594133,0.594133,0.594133,0.594133
lag_2,0.102322,0.102322,0.102322,0.102322,0.102322
lag_21,0.051588,0.051588,0.051588,0.051588,0.051588
lag_3,0.051227,0.051227,0.051227,0.051227,0.051227
lag_22,0.049085,0.049085,0.049085,0.049085,0.049085
lag_18,0.041700,0.041700,0.041700,0.041700,0.041700
lag_20,0.040177,0.040177,0.040177,0.040177,0.040177
lag_8,0.039240,0.039240,0.039240,0.039240,0.039240
lag_19,0.028528,0.028528,0.028528,0.028528,0.028528
lag_23,0.021719,0.021719,0.021719,0.021719,0.021719


In [22]:
# Feature ranking stability across stores
stability = hierarchical.feature_ranking_stability(
    X, level="store", top_n=10, feature_names=feature_names
)
print("Feature Ranking Stability:")
stability

Feature Ranking Stability:


,mean_rank,std_rank,min_rank,max_rank
lag_1,1.0,0.0,1.0,1.0
lag_2,2.0,0.0,2.0,2.0
lag_21,3.0,0.0,3.0,3.0
lag_3,4.0,0.0,4.0,4.0
lag_22,5.0,0.0,5.0,5.0
lag_18,6.0,0.0,6.0,6.0
lag_20,7.0,0.0,7.0,7.0
lag_8,8.0,0.0,8.0,8.0
lag_19,9.0,0.0,9.0,9.0
lag_23,10.0,0.0,10.0,10.0


In [23]:
# Heatmap of importance across stores
from xeries.visualization import plot_hierarchy_heatmap

fig, ax = plot_hierarchy_heatmap(hier_result, level="store", top_n=15)
plt.tight_layout()
plt.show()

## 11. Using KernelExplainer with Series-Specific Background

For non-tree models or when you want series-specific baselines, use `explainer_type='kernel'`.

**Note:** KernelExplainer is much slower than TreeExplainer, so we'll only compute for a small subset.

In [24]:
# Force KernelExplainer with series-specific background
kernel_explainer = ConditionalSHAP(
    model=adapter.forecaster.estimator,
    background_data=X,
    series_col="_level_skforecast",
    explainer_type="kernel",
    background_strategy="series",  # Use series-specific background
    n_background_samples=30,
)

print(f"Batch capable: {kernel_explainer._is_batch_capable}")
print(f"Series backgrounds prepared: {len(kernel_explainer._series_backgrounds)}")

Batch capable: False
Series backgrounds prepared: 5


In [25]:
# Compute SHAP for a small subset (KernelExplainer is slow)
X_small = X.groupby("_level_skforecast").head(3)  # 3 samples per series
print(f"Computing SHAP for {len(X_small)} samples...")

try:
    kernel_result = kernel_explainer.explain(X_small, feature_names=feature_names)
    print(f"\nSHAP values shape: {kernel_result.shap_values.shape}")
    print("\nGlobal importance (KernelExplainer):")
    display(kernel_result.mean_abs_shap().head(10))
except Exception as exc:
    print(
        "Skipping KernelExplainer demo "
        f"({type(exc).__name__}: {exc}). TreeExplainer results above are the primary path."
    )

Computing SHAP for 15 samples...
Skipping KernelExplainer demo (AttributeError: property 'feature_names_in_' of 'LGBMRegressor' object has no setter). TreeExplainer results above are the primary path.


## Summary

The enhanced `ConditionalSHAP` provides:

1. **Auto-detection** of model type for optimal explainer selection (TreeExplainer for LightGBM)
2. **Batch computation** for tree models (fast)
3. **Series-specific background** for kernel explainer (when needed)
4. **`explain_per_series()`** for detailed per-series analysis
5. **Integration with `HierarchicalExplainer`** for multi-level aggregation
6. **Full evaluation support** - entire dataset, no sampling required
7. **Visualization** at each hierarchy level (beeswarm, bar, heatmap)